In [11]:
from pathlib import Path
import sys

project_root=Path.cwd().parents[1]
sys.path.append(str(project_root))

In [12]:
from src.retrieval.base import BaseRetriever
from src.embeddings.embedding_manager import EmbeddingManager
from src.vector_db.vector_store import VectorStore
from src.retrieval.models import (RetrievalResult, RetrievalRequest, RetrievedDocument, RetrievalStrategy, SearchCandidate)
import numpy as np
from sentence_transformers import CrossEncoder

from src.retrieval.cross_encoder_reranker import CrossEncoderReranker



from chromadb.api.types import QueryResult
import time
from typing import Any

In [14]:
class DenseRetriever(BaseRetriever):
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore, cross_encoder:CrossEncoderReranker):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store
        self.cross_encoder=cross_encoder

    def _validate_request(self, request: RetrievalRequest):
        
        if request.query is None:
            raise ValueError(" there is no query give a query atleast")

        if not request.query.strip():
            raise ValueError(" there is no query give a query atleast")
        
        if ( request.top_k <= 0):
            raise ValueError("invalid topk argument")
        
        if request.strategy is None:
            raise ValueError(" please enter a valid startergy of retiever")
        
    def _embed_query(self, query: str)->np.ndarray:
        embeddings=self.embedding_manager.generate_embeddings(texts=[query])
        return embeddings

    def _search(self, embeddings:np.ndarray, top_k:int)->QueryResult:
        
        search_results=self.vector_store.query( query_embedding =embeddings, top_k=top_k )
        return search_results

    def _convert_candidates(self, search_results:QueryResult)->list[SearchCandidate]:

        retrieve_docs: list[SearchCandidate]=[]
        ids = search_results["ids"]
        documents = search_results["documents"]
        metadatas = search_results["metadatas"]
        distances = search_results["distances"]

        if documents is None or metadatas is None or distances is None:
            raise ValueError("QueryResult is missing requested fields.")

        for ids_batch, documents_batch, metadatas_batch, distances_batch in zip(
        ids,
        documents,
        metadatas,
        distances
        ):
            
            for rank, (chunk_id, document, metadata, distance) in enumerate(zip(ids_batch, documents_batch, metadatas_batch, distances_batch), start=1):
                document_id=metadata.get("document_id")
                if document_id is None:
                    raise ValueError(" no document id")
                
                retrieve_docs.append(
                SearchCandidate(
                    document_id=document_id,
                    chunk_id=chunk_id,
                    text= document,
                    metadata=metadata,
                    distance=distance,
                    retrieval_rank=rank,
                    final_rank=None,
                    reranker_score=None
                )
            )
        
        return retrieve_docs


    def _sort_candidates(self, candidates:list[SearchCandidate])-> list[SearchCandidate]:

        if any(candidate.reranker_score is None for candidate in candidates):
            raise ValueError("All candidates must have a raw_score before sorting.")

        return sorted(candidates, key=lambda candidate: candidate.reranker_score if candidate.reranker_score is not None else float("-inf"), reverse=True )

    
    def _apply_threshold(self, candidates:list[SearchCandidate], threshold:float|None)->list[SearchCandidate]:

        if threshold is None:
            return candidates

        return [ candidate for candidate in candidates if candidate.distance is not None and candidate.distance < threshold ]

    def _apply_filters(self,candidates:list[SearchCandidate], filters:dict[str, Any] | None = None)->list[SearchCandidate]:

        if filters is None:
            return candidates

        return candidates


    def _remove_duplicates(
    self,
    candidates: list[SearchCandidate],
    ) -> list[SearchCandidate]:

        seen = set()
        unique:list[SearchCandidate] = []

        for candidate in candidates:
            if candidate.chunk_id not in seen:
                seen.add(candidate.chunk_id)
                unique.append(candidate)
        return unique    
    
    def _assign_rank(self, candidates:list[SearchCandidate]) ->list[SearchCandidate] :

        for rank,candaidate in enumerate(candidates, start=1):
            candaidate.final_rank=rank
        
        return candidates

    def _process_candidates(self,query:str, candidates:list[SearchCandidate], threshold:float|None, filters:dict[str, Any] | None = None )->list[SearchCandidate]:

            candidates=self._apply_threshold(candidates, threshold)

            candidates=self._apply_filters(candidates, filters=filters)

            candidates=self._remove_duplicates(candidates)

            candidates=self.cross_encoder.rerank(query=query,documents=candidates )

            candidates=self._sort_candidates(candidates)

            candidates=self._assign_rank(candidates)

            return candidates

    def _build_retrieved_document(self, candidate_results:list[SearchCandidate])->list[RetrievedDocument]:
        
        docs:list[RetrievedDocument]=[]

        for doc in candidate_results:
            docs.append(
                RetrievedDocument(
                    document_id=doc.document_id,
                    chunk_id=doc.chunk_id,
                    content=doc.text,
                    metadata=doc.metadata,
                    retrieval_rank=doc.retrieval_rank,
                    final_rank=doc.final_rank,
                    distance=doc.distance,
                    reranker_score=doc.reranker_score
                )
            )
        return docs
    
    def _build_result( self, retrieved_docs:list[RetrievedDocument], query:str, strategy:RetrievalStrategy, elapsed:float)->RetrievalResult:
        
        return RetrievalResult(
            query=query,
            strategy=strategy,
            retrieval_time=elapsed,
            retrieved_documents=retrieved_docs
        )

    def retrieve(self, request: RetrievalRequest) -> RetrievalResult:
        
        start=time.perf_counter()

        self._validate_request(request=request)

        embeddings=self._embed_query(
            query= request.query
        )

        search_results=self._search(
            embeddings=embeddings,
            top_k=request.top_k 
        )

        candidates=self._convert_candidates(
            search_results=search_results
        )

        candidate_results=self._process_candidates(
            query=request.query,
            candidates=candidates,
            threshold=request.score_threshold
        )

        retrieved_docs=self._build_retrieved_document(
            candidate_results=candidate_results
        )
        elapsed=time.perf_counter() - start
        result=self._build_result(
            retrieved_docs=retrieved_docs,
            query=request.query,
            strategy=request.strategy,
            elapsed=elapsed
        )    

        return result